# 🎨 Voice Designer Foundry MVP — Qwen3-TTS
A prompt engineering studio for crafting custom voice personas using the VoiceDesign model.

**Examples of Voice Design Prompts:**
- "A deep, authoritative older male documentary narrator with a British accent speaking slowly."
- "A bubbly, fast-talking young female podcast host with a Californian accent."
- "A raspy, menacing villain with a slow, deliberate cadence."


In [ ]:
!pip install -q qwen-tts soundfile gradio

import os
import torch
import soundfile as sf
import gradio as gr
from IPython.display import Audio, display


In [ ]:
# VoiceDesign is ALWAYS 1.7B — no 0.6B variant exists
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
print(f'Using Model: {MODEL_ID}')


In [ ]:
from qwen_tts.model import Qwen3TTSModel

def to_wav(result, default_sr=24000):
    """Normalize any Qwen3-TTS generate_* return into (waveform, sample_rate)."""
    audio, sr = result if isinstance(result, tuple) else (result, default_sr)
    if isinstance(audio, (list, tuple)):
        audio = audio[0]
    if hasattr(audio, "cpu"):
        audio = audio.cpu().numpy()
    return audio, sr

print(f'Loading model {MODEL_ID}...')
vd_model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation='sdpa'
)
print('Model loaded successfully!')


In [ ]:
VOICE_ATTRIBUTES = {
    'gender': 'female',
    'age_range': 'young',
    'tone': 'warm',
    'pace': 'moderate',
    'accent': 'American',
    'emotion': 'cheerful',
    'style': 'conversational'
}

def build_voice_prompt(attrs):
    return f"A {attrs.get('age_range', '')} {attrs.get('gender', '')} voice with an {attrs.get('accent', '')} accent. The tone is {attrs.get('tone', '')} and {attrs.get('emotion', '')}. They are speaking at a {attrs.get('pace', '')} pace in a {attrs.get('style', '')} style."

print('Example Personas:')
print('1.', build_voice_prompt(VOICE_ATTRIBUTES))
print('2.', build_voice_prompt({'gender': 'male', 'age_range': 'middle-aged', 'tone': 'authoritative', 'pace': 'slow', 'accent': 'British'}))
print('3.', build_voice_prompt({'gender': 'female', 'age_range': 'child', 'tone': 'playful', 'pace': 'fast', 'accent': 'Australian'}))


In [ ]:
prompt = build_voice_prompt(VOICE_ATTRIBUTES)
text = 'Welcome to the Voice Designer Foundry. I can be customized to sound like almost anyone.'

print(f'Prompt: {prompt}')
audio = vd_model.generate_voice_design(text=text, language='en', instruct=prompt)
audio, sr = to_wav(audio)
sf.write('single_voice.wav', audio, sr)
display(Audio('single_voice.wav'))


In [ ]:
prompts = {
    'a': 'A deep, authoritative older male documentary narrator with a British accent speaking slowly.',
    'b': 'A bubbly, fast-talking young female podcast host with a Californian accent.',
    'c': 'A raspy, menacing villain with a slow, deliberate cadence.'
}

test_text = 'The secret to this entire process is hidden deep within the ancient ruins.'

for key, prompt in prompts.items():
    print(f'\nGenerating Voice {key.upper()}...\nPrompt: {prompt}')
    audio = vd_model.generate_voice_design(text=test_text, language='en', instruct=prompt)
    filename = f'voice_{key}.wav'
    audio, sr = to_wav(audio)
    sf.write(filename, audio, sr)
    display(Audio(filename))


In [ ]:
portfolio_prompts = {
    'wise_elder': 'An old wise female speaking slowly with a calming, gentle tone.',
    'energetic_youth': 'A hyperactive young boy speaking very quickly and excitedly.',
    'calm_monk': 'A serene male monk speaking with a steady, peaceful cadence.',
    'dramatic_actor': 'A theatrical male actor speaking with intense emotion and dynamic range.',
    'robotic_ai': 'A female AI assistant speaking with a flat, slightly robotic, but polite tone.'
}

portfolio_text = 'Hello, traveler. It is a pleasure to meet you.'

for name, prompt in portfolio_prompts.items():
    print(f'\nPortfolio: {name}\nPrompt: {prompt}')
    audio = vd_model.generate_voice_design(text=portfolio_text, language='en', instruct=prompt)
    filename = f'portfolio_{name}.wav'
    audio, sr = to_wav(audio)
    sf.write(filename, audio, sr)
    display(Audio(filename))


## Export as Reference Clips
You can use the WAV files generated above as `ref_audio` in the **Voice Clone Studio** (Notebook 03) to clone these exact voices! Just download the WAV files and upload them there.


In [ ]:
def gen_single(instruct, text):
    try:
        audio = vd_model.generate_voice_design(text=text, language='en', instruct=instruct)
        filename = 'gradio_vd.wav'
        audio, sr = to_wav(audio)
        sf.write(filename, audio, sr)
        return filename
    except Exception as e:
        print(f'Error: {e}')
        return None

def gen_abc(prompt_a, prompt_b, prompt_c, text):
    out_a = gen_single(prompt_a, text) if prompt_a else None
    out_b = gen_single(prompt_b, text) if prompt_b else None
    out_c = gen_single(prompt_c, text) if prompt_c else None
    return out_a, out_b, out_c

with gr.Blocks() as demo:
    gr.Markdown('# 🎨 Voice Designer Foundry')
    with gr.Tabs():
        with gr.TabItem('Single Design'):
            t1 = gr.Textbox(label='Voice Description (Instruct)')
            t2 = gr.Textbox(label='Text to Say')
            b1 = gr.Button('Generate')
            a1 = gr.Audio(label='Output')
            b1.click(gen_single, inputs=[t1, t2], outputs=[a1])
        with gr.TabItem('A/B/C Audition'):
            t_text = gr.Textbox(label='Text to Say')
            with gr.Row():
                ta = gr.Textbox(label='Voice A Description')
                tb = gr.Textbox(label='Voice B Description')
                tc = gr.Textbox(label='Voice C Description')
            b_abc = gr.Button('Generate All')
            with gr.Row():
                aa = gr.Audio(label='Voice A Output')
                ab = gr.Audio(label='Voice B Output')
                ac = gr.Audio(label='Voice C Output')
            b_abc.click(gen_abc, inputs=[ta, tb, tc, t_text], outputs=[aa, ab, ac])

demo.launch(share=True, debug=True)
